<a href="https://colab.research.google.com/github/syedmahmoodiagents/computervision/blob/main/Penn_Fudan_Ped.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from torchvision import transforms

In [2]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset


In [ ]:
import os

In [ ]:
!wget https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip
!unzip PennFudanPed.zip

--2026-01-25 03:08:49--  https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip
Resolving www.cis.upenn.edu (www.cis.upenn.edu)... 158.130.69.163, 2607:f470:8:64:5ea5::d
Connecting to www.cis.upenn.edu (www.cis.upenn.edu)|158.130.69.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53723336 (51M) [application/zip]
Saving to: ‘PennFudanPed.zip’

PennFudanPed.zip    100%[===================>]  51.23M   130MB/s    in 0.4s    

2026-01-25 03:08:49 (130 MB/s) - ‘PennFudanPed.zip’ saved [53723336/53723336]

Archive:  PennFudanPed.zip
   creating: PennFudanPed/
  inflating: PennFudanPed/added-object-list.txt  
   creating: PennFudanPed/Annotation/
  inflating: PennFudanPed/Annotation/FudanPed00001.txt  
  inflating: PennFudanPed/Annotation/FudanPed00002.txt  
  inflating: PennFudanPed/Annotation/FudanPed00003.txt  
  inflating: PennFudanPed/Annotation/FudanPed00004.txt  
  inflating: PennFudanPed/Annotation/FudanPed00005.txt  
  inflating: PennFudanPed/Annotatio

In [10]:
num_images = len(os.listdir("PennFudanPed/PNGImages"))
num_masks  = len(os.listdir("PennFudanPed/PedMasks"))

print(num_images, num_masks)


170 170


In [1]:


def masks_to_boxes(mask):
    obj_ids = np.unique(mask)
    obj_ids = obj_ids[1:]

    boxes = []
    for obj_id in obj_ids:
        pos = np.where(mask == obj_id)
        xmin, xmax = pos[1].min(), pos[1].max()
        ymin, ymax = pos[0].min(), pos[0].max()
        boxes.append([xmin, ymin, xmax, ymax])

    return torch.tensor(boxes, dtype=torch.float32)


In [3]:
class PennFudanDataset(Dataset):
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms

        self.imgs = sorted(os.listdir(os.path.join(root, "PNGImages")))
        self.masks = sorted(os.listdir(os.path.join(root, "PedMasks")))

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):

        img_path = os.path.join(self.root, "PNGImages", self.imgs[idx])
        mask_path = os.path.join(self.root, "PedMasks", self.masks[idx])

        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        mask = np.array(mask)

        # Extract object ids
        obj_ids = np.unique(mask)
        obj_ids = obj_ids[1:]  # remove background

        num_objs = len(obj_ids)

        # Convert masks to bounding boxes
        boxes = []
        for obj_id in obj_ids:
            pos = np.where(mask == obj_id)
            xmin = np.min(pos[1])
            xmax = np.max(pos[1])
            ymin = np.min(pos[0])
            ymax = np.max(pos[0])
            boxes.append([xmin, ymin, xmax, ymax])

        boxes = torch.tensor(boxes, dtype=torch.float32)

        # Labels (only 1 class: pedestrian)
        labels = torch.ones((num_objs,), dtype=torch.int64)

        # Area
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])

        # Crowd flag (all 0 here)
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)

        # Final target dict (torchvision standard)
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": iscrowd,
        }

        if self.transforms is not None:
            img = self.transforms(img)

        return img, target


In [12]:
import torchvision.transforms as T

In [4]:
def get_transform(train=True):
    transforms = []
    transforms.append(T.ToTensor())
    return T.Compose(transforms)


In [11]:
from torch.utils.data import DataLoader

In [5]:
def collate_fn(batch):
    return tuple(zip(*batch))


In [6]:
dataset = PennFudanDataset(
    root="PennFudanPed",
    transforms=get_transform()
)

In [9]:
len(dataset)

170

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)


In [ ]:
images, targets = next(iter(loader))

In [7]:
print(len(images))              # batch size
print(images[0].shape)          # [3, H, W]
print(targets[0].keys())
print(targets[0]["boxes"])


2
torch.Size([3, 376, 508])
dict_keys(['boxes', 'labels', 'image_id', 'area', 'iscrowd'])
tensor([[ 91.,  61., 235., 343.],
        [241.,  51., 300., 354.]])


In [8]:
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn

model = fasterrcnn_resnet50_fpn(num_classes=2)  # background + person

images, targets = next(iter(loader))
loss_dict = model(images, targets)
loss = sum(loss for loss in loss_dict.values())

print(loss.item())


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 125MB/s]


1.486759066581726


In [13]:
# from torch.optim import SGD

In [14]:
# def get_transform(train=True):
#     return T.Compose([T.ToTensor()])
